In [1]:
from load_filtered_dataset import *

/usr/local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_filtered_dataset()['train']

Loading dataset with 23124 kept samples (out of 25443 total)


In [3]:
ds[0]

{'question': 'There are $n$ candy boxes in front of Tania. The boxes are arranged in a row from left to right, numbered from $1$ to $n$. The $i$-th box contains $r_i$ candies, candies have the color $c_i$ (the color can take one of three values \u200b\u200b— red, green, or blue). All candies inside a single box have the same color (and it is equal to $c_i$).\n\nInitially, Tanya is next to the box number $s$. Tanya can move to the neighbor box (that is, with a number that differs by one) or eat candies in the current box. Tanya eats candies instantly, but the movement takes one second.\n\nIf Tanya eats candies from the box, then the box itself remains in place, but there is no more candies in it. In other words, Tanya always eats all the candies from the box and candies in the boxes are not refilled.\n\nIt is known that Tanya cannot eat candies of the same color one after another (that is, the colors of candies in two consecutive boxes from which she eats candies are always different). 

# Statistics

In [28]:
from tqdm import tqdm
difficulties = []
solution_cnts = []
skills = []

for sample in tqdm(ds):
    if 'difficulty' in sample:
        difficulties.append(sample['difficulty'])
    else:
        difficulties.append('unknown')
    if 'solutions' in sample:
        solution_cnts.append(len(eval(sample['solutions'])))
    else:
        solution_cnts.append(0)
    if 'skill_types' in sample:
        for skill in eval(sample['skill_types']):
            skills.append(skill)

100%|██████████| 23124/23124 [00:46<00:00, 498.41it/s]


In [29]:
import pandas as pd

difficulties_series = pd.Series(difficulties, name="difficulty")
solution_cnts_series = pd.Series(solution_cnts, name='solution_cnts')
skills_series = pd.Series(skills, name='skills')

viz_difficulty_order = {'UNKNOWN_DIFFICULTY': 0, 'EASY': 1, 'MEDIUM': 2, 'MEDIUM_HARD': 3, 'HARD': 4, 'VERY_HARD': 5}

# Create a DataFrame to allow adding a column for difficulty level
difficulties_df = difficulties_series.to_frame()
difficulties_df["difficulty_level"] = difficulties_df["difficulty"].map(viz_difficulty_order).fillna(-1).astype(int)
difficulties_df = difficulties_df.sort_values("difficulty_level")

difficulty_counts = difficulties_df["difficulty"].value_counts().reindex(list(viz_difficulty_order.keys()), fill_value=0)
solution_counts = solution_cnts_series.value_counts().sort_index()

skills_counts = skills_series.value_counts()
display("### Difficulty", difficulty_counts)
display("### Solution number", solution_counts)
display("### Skills", skills_counts)

'### Difficulty'

difficulty
UNKNOWN_DIFFICULTY    4450
EASY                  8762
MEDIUM                3086
MEDIUM_HARD           2578
HARD                  2606
VERY_HARD             1642
Name: count, dtype: int64

'### Solution number'

solution_cnts
0       4647
1       1216
2        719
3        555
4        482
        ... 
5761       1
5885       1
5893       1
6372       1
6391       1
Name: count, Length: 824, dtype: int64

'### Skills'

skills
Data structures        4458
Greedy algorithms      2463
Dynamic programming    2219
Sorting                2024
Complete search        1712
Bit manipulation        730
Amortized analysis      526
Range queries           206
Name: count, dtype: int64

# Visualization

In [34]:
import random

def sample_problem_by_difficulty(ds, difficulty="HARD"):
    """
    Efficiently sample a random problem from the dataset with the given difficulty level.
    Returns the sample dict.
    """
    # Avoid collecting all into a list; make one pass and pick randomly (reservoir sampling)
    count = 0
    chosen = None
    target_difficulty = difficulty.upper()
    for s in ds:
        sample_difficulty = s.get("difficulty", "unknown").upper()
        if sample_difficulty == target_difficulty:
            count += 1
            if random.randint(1, count) == 1:
                chosen = s
    if chosen is None:
        raise ValueError(f"No problems found for difficulty '{difficulty}'.")
    return chosen

def format_problem(sample):
    """Format a problem for easy visualization."""
    question = sample.get("question", "(no question available)")
    url = sample.get("url", "(no URL)")
    title = sample.get("name", "(no title)")
    skill_types = eval(sample.get("skill_types", "[]"))
    return f"""## {title}
Difficulty: {sample.get("difficulty", "unknown")}
Skills: {', '.join(skill_types)}
URL: {url}

**Problem Statement:**  
{question}
"""

def format_solutions(sample):
    """Format all solutions for display."""
    solutions = eval(sample.get("solutions", "[]"))
    if not solutions:
        return "No solutions found."
    if len(solutions) > 1:
        solutions = solutions[:1]
    return "\n\n".join([f"### Solution {i+1}\n```python\n{sol}\n```" for i, sol in enumerate(solutions)])

# Usage example:
# sampled = sample_problem_by_difficulty(ds, difficulty="HARD")
# print(format_problem(sampled))
# print(format_solutions(sampled))

In [35]:
sampled = sample_problem_by_difficulty(ds, difficulty="EASY")
print(format_problem(sampled))
print(format_solutions(sampled))

## None
Difficulty: EASY
Skills: 
URL: https://www.codewars.com/kata/589ebcb9926baae92e000001

**Problem Statement:**  
You have to create a function that converts integer given as string into ASCII uppercase letters.

All ASCII characters have their numerical order in table. 

For example,

```
from ASCII table, character of number 65 is "A".
```

Numbers will be next to each other, So you have to split given number to two digit long integers.

For example, 

```
'658776' to [65, 87, 76] and then turn it into 'AWL'.
```

### Solution 1
```python
def convert(number):
	return ''.join((chr(int(number[a:a + 2])) for a in range(0, len(number), 2)))

```


In [36]:
sampled = sample_problem_by_difficulty(ds, difficulty="UNKNOWN_DIFFICULTY")
print(format_problem(sampled))
print(format_solutions(sampled))

## None
Difficulty: UNKNOWN_DIFFICULTY
Skills: 
URL: https://www.codechef.com/problems/DECODEIT

**Problem Statement:**  
An encoder encodes the first $16$ lowercase English letters using $4$ bits each. The first bit (from the left) of the code is $0$ if the letter lies among the first $8$ letters, else it is $1$, signifying that it lies among the last $8$ letters. The second bit of the code is $0$ if the letter lies among the first $4$ letters of those $8$ letters found in the previous step, else it's $1$, signifying that it lies among the last $4$ letters of those $8$ letters. Similarly, the third and the fourth bit each signify the half in which the letter lies. 
For example, the letter $j$ would be encoded as :
- Among $(a,b,c,d,e,f,g,h$ $|$ $i,j,k,l,m,n,o,p)$, $j$ appears in the second half. So the first bit of its encoding is $1$.
- Now, among $(i,j,k,l$ $|$ $m,n,o,p)$, $j$ appears in the first half. So the second bit of its encoding is $0$.
- Now, among $(i,j$ $|$ $k,l)$, $j$ ap

In [37]:
sampled = sample_problem_by_difficulty(ds, difficulty="UNKNOWN_DIFFICULTY")
print(format_problem(sampled))
print(format_solutions(sampled))

## None
Difficulty: UNKNOWN_DIFFICULTY
Skills: Greedy algorithms
URL: https://codeforces.com/problemset/problem/1297/C

**Problem Statement:**  
Polycarp is the project manager in the IT-company. Right now, he needs to choose developers for his team to start a new project. The company has n developers "on the bench" (i.e not involved in other projects). Polycarp assessed the skills of each of them: a_i (-10^4 ≤ a_i ≤ 10^4) — an integer characteristic of the i-th developer. This value can be either positive, zero or even negative (some developers cause distractions).

After Polycarp chooses a subset of developers for his team, the strength of the team will be determined by the sum of a_i values for all selected developers.

Polycarp fears that if he chooses a team in such a way that maximizes the sum of the characteristics of a_i, other managers may find this unacceptable. For this reason, he plans to create such a team that the sum of the a_i values for it is strictly less than the max

In [38]:
sampled = sample_problem_by_difficulty(ds, difficulty="MEDIUM")
print(format_problem(sampled))
print(format_solutions(sampled))

## None
Difficulty: MEDIUM
Skills: Dynamic programming
URL: https://practice.geeksforgeeks.org/problems/longest-common-subsequence-1587115620/1

**Problem Statement:**  
Given two sequences, find the length of longest subsequence present in both of them. Both the strings are of uppercase.
Example 1:
Input:
A = 6, B = 6
str1 = ABCDGH
str2 = AEDFHR
Output: 3
Explanation: LCS for input Sequences
“ABCDGH” and “AEDFHR” is “ADH” of
length 3.
Example 2:
Input:
A = 3, B = 2
str1 = ABC
str2 = AC
Output: 2
Explanation: LCS of "ABC" and "AC" is
"AC" of length 2.
Your Task:
Complete the function lcs() which takes the length of two strings respectively and two strings as input parameters and returns the length of the longest subsequence present in both of them. 
Expected Time Complexity : O(|str1|*|str2|)
Expected Auxiliary Space: O(|str1|*|str2|)
Constraints:
1<=size(str1),size(str2)<=10^{3}

### Solution 1
```python
class Solution:

	def fun(self, s1, s2, x, y):
		maxi = 0
		l = [[0 for p in rang